# Tugas 1 - Data Wrangling
Integrasi 3 sumber data: CSV (POS), SQLite (CRM), JSON (API logistik).
Dataset transaksi dari UCI Online Retail, data pelanggan & logistik dibuat mengikuti transaksinya.

In [1]:
import pandas as pd
import numpy as np

df_pos = pd.read_csv('transaksi_pos_nrm.csv', parse_dates=['InvoiceDate'],
                     dtype={'InvoiceNo': str, 'StockCode': str})
df_pos.shape

(80000, 11)

## B.2 Database CRM

In [2]:
from sqlalchemy import create_engine

engine = create_engine('sqlite:///customers_nrm.db')
df_crm = pd.read_sql("SELECT * FROM customers WHERE is_active = 1", engine)
df_crm.shape

(1755, 8)

## B.3 REST API logistik
Endpoint aslinya nggak publik, jadi responsenya di-mock jadi file JSON dengan struktur sama. JSON-nya bersarang, diratakan pakai json_normalize.

In [3]:
import requests, json

try:
    resp = requests.get('https://raw.githubusercontent.com/adamhasani/tugas1-data-wrangling/master/status_pengiriman.json', timeout=30)
    resp.raise_for_status()
    api_data = resp.json()
except Exception:
    with open('status_pengiriman.json') as f:
        api_data = json.load(f)

df_logistik = pd.json_normalize(api_data['data'], sep='_')
df_logistik.shape

(4005, 8)

## B.4 Merge
Join pakai LEFT karena transaksi tanpa record pengiriman (belum diproses) dan pelanggan non-aktif tetap harus ikut dihitung. Logistik di-dedup dulu karena API-nya level order, sedangkan POS level item.

In [4]:
df_logistik = df_logistik.drop_duplicates(subset='order_id')
df_pos = df_pos.drop_duplicates(subset=['InvoiceNo', 'StockCode'])

df_master_analisis = df_pos.merge(df_logistik, how='left',
                                  left_on='InvoiceNo', right_on='order_id')
df_master_analisis = df_master_analisis.merge(df_crm, how='left', on='customer_id')
df_master_analisis = df_master_analisis.drop(columns=['order_id'])
df_master_analisis.shape

(78035, 25)

In [5]:
# jawaban pertanyaan data owner
kupon = df_master_analisis[df_master_analisis['coupon_code'].notna()]
inv = kupon.drop_duplicates(subset='InvoiceNo')
print('total transaksi kupon :', len(inv), 'invoice')
print('total nilai           :', kupon['total_amount'].sum().round(2))
print()
print(inv['membership_tier'].value_counts())
print()
inv_kirim = inv.dropna(subset=['status'])
terkirim = (inv_kirim['status'] == 'Delivered').sum()
print(f'terkirim: {terkirim}/{len(inv_kirim)} ({terkirim/len(inv_kirim)*100:.1f}%)')

total transaksi kupon : 203 invoice
total nilai           : 68949.57

membership_tier
Silver      51
Bronze      49
Gold        46
Platinum    43
Name: count, dtype: int64

terkirim: 153/203 (75.4%)


## C.1 Kamus Data

In [6]:
def generate_data_dictionary(df):
    kamus = pd.DataFrame({
        'nama_kolom'      : df.columns,
        'tipe_data_sistem': [str(t) for t in df.dtypes],
        'jumlah_missing'  : df.isna().sum().values,
        'persen_missing'  : (df.isna().mean().values * 100).round(2),
        'jumlah_nilai_unik': df.nunique(dropna=True).values,
    })
    kamus['sampel_nilai'] = [', '.join(str(v) for v in df[c].dropna().unique()[:3]) for c in df.columns]
    return kamus

kamus = generate_data_dictionary(df_master_analisis)
kamus.to_csv('kamus_data_hasil.csv', index=False)
kamus

,nama_kolom,tipe_data_sistem,jumlah_missing,persen_missing,jumlah_nilai_unik,sampel_nilai
0,InvoiceNo,str,0,0.00,4005,"536365, 536366, 536367"
1,StockCode,str,0,0.00,2832,"85123A, 71053, 84406B"
2,Description,str,0,0.00,2860,"WHITE HANGING HEART T-LIGHT HOLDER, WHITE META..."
3,Quantity,int64,0,0.00,154,"6, 8, 2"
4,InvoiceDate,datetime64[us],0,0.00,3796,"2010-12-01 08:26:00, 2010-12-01 08:28:00, 2010..."
5,UnitPrice,float64,0,0.00,188,"2.55, 3.39, 2.75"
6,CustomerID,int64,0,0.00,1904,"17850, 13047, 12583"
7,Country,str,0,0.00,30,"United Kingdom, France, Australia"
8,total_amount,float64,0,0.00,1229,"15.3, 20.34, 22.0"
9,coupon_code,str,73988,94.81,4,"NRMFLASH15, AKHIRBULAN25, DISKON20"


## C.2 Validasi

In [7]:
assert df_master_analisis.duplicated(subset=['InvoiceNo', 'StockCode']).sum() == 0
assert (df_master_analisis['total_amount'] >= 0).all()
assert (df_master_analisis['InvoiceDate'] <= pd.Timestamp.now()).all()
print('semua validasi lolos')

semua validasi lolos
